# 06b: Train & Evaluate the Decoder

**Runtime: T4 (free tier)** — GPT-2 Medium is only 1.4 GB

Picks up training data from Google Drive (generated by 06a on A100),
fine-tunes GPT-2 Medium with LoRA, evaluates quality.

**Time:** ~2 hrs on T4 | **Cost:** $0

In [ ]:
!pip install -q peft datasets sentence-transformers
!nvidia-smi | head -4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/HLC')

In [ ]:
from pathlib import Path
from hlc.config import Config
from hlc.decoder_training import DecoderTrainer

config = Config()
trainer = DecoderTrainer(config)

DATA_DIR = Path('/content/drive/MyDrive/HLC/data')
TRAINING_DATA = DATA_DIR / 'decoder_training.jsonl'

# Verify training data exists
assert TRAINING_DATA.exists(), f'Training data not found at {TRAINING_DATA}. Run 06a first!'
import json
line_count = sum(1 for _ in open(TRAINING_DATA))
print(f'Training data: {line_count} examples')
print(f'File size: {TRAINING_DATA.stat().st_size / 1024 / 1024:.1f} MB')
print(f'Decoder base model: {config.decoder_model_name}')

## Step 1: LoRA Fine-tune (~2 hrs on T4, ~45 min on A100)

In [ ]:
trainer.train(
    training_data_path=TRAINING_DATA,
    epochs=3,
    batch_size=4,
    learning_rate=2e-4,
    lora_rank=16,
    lora_alpha=32,
)

## Step 2: Evaluate

In [ ]:
from hlc.decoder import Decoder
from hlc.routing import RoutingResult
from hlc.value_system import ValueState
import torch

decoder = Decoder(config)

# === Basic test ===
test_result = RoutingResult(
    converged=True, iterations=1,
    final_state=torch.randn(384),
    active_column_ids=['test-1'],
    active_source_texts=['Water boils at 100 degrees Celsius at standard atmospheric pressure.'],
    prediction_error=0.05,
    value_state=ValueState(joy=0.8, curiosity=0.1),
    mode='fast',
)

response = decoder.generate(test_result, 'What temperature does water boil at?')
print(f'Q: What temperature does water boil at?')
print(f'R: {response}')

In [ ]:
# === Critical Test: False Fact ===
# Decoder should say "cheese" — proves it has no internal knowledge
false_result = RoutingResult(
    converged=True, iterations=1,
    final_state=torch.randn(384),
    active_column_ids=['false-1'],
    active_source_texts=['The sun is made of cheese.'],
    prediction_error=0.05,
    value_state=ValueState(joy=0.8),
    mode='fast',
)

response = decoder.generate(false_result, 'What is the sun made of?')
print(f'Q: What is the sun made of?')
print(f'K: The sun is made of cheese.')
print(f'R: {response}')
print()
print('PASS' if 'cheese' in response.lower() else 'FAIL')

In [ ]:
# === Empty Knowledge Test ===
empty_result = RoutingResult(
    converged=False, iterations=8,
    final_state=torch.randn(384),
    active_column_ids=[], active_source_texts=[],
    prediction_error=0.9,
    value_state=ValueState(curiosity=0.6, pain=0.2),
    mode='slow',
)

response = decoder.generate(empty_result, 'How do I bake a chocolate cake?')
print(f'Q: How do I bake a chocolate cake?')
print(f'K: (none)')
print(f'R: {response}')

In [ ]:
# === Batch Evaluation ===
test_cases = [
    ('What is DNA?', ['DNA contains the genetic instructions for the development and function of living organisms.'], 0.8, 0.1),
    ('How fast does light travel?', ['Light travels at approximately 300,000 kilometers per second in a vacuum.'], 0.9, 0.05),
    ('What is Pi?', ['Pi is approximately 3.14159 and represents the ratio of a circle\'s circumference to its diameter.'], 0.7, 0.1),
    ('What are cells?', [
        'All living organisms are made of cells, which are the basic units of life.',
        'Bacteria are single-celled organisms that can be beneficial or harmful to other living things.',
    ], 0.6, 0.2),
    ('What is quantum computing?', [], 0.0, 0.8),
]

print('=== Batch Evaluation ===')
for query, knowledge, joy, curiosity in test_cases:
    result = RoutingResult(
        converged=bool(knowledge),
        iterations=1 if knowledge else 8,
        final_state=torch.randn(384),
        active_column_ids=[f'col-{i}' for i in range(len(knowledge))],
        active_source_texts=knowledge,
        prediction_error=0.1 if knowledge else 0.9,
        value_state=ValueState(joy=joy, curiosity=curiosity),
        mode='fast' if joy > 0.7 else 'slow',
    )
    response = decoder.generate(result, query)
    print(f'\nQ: {query}')
    print(f'K: {knowledge[0][:60]}...' if knowledge else 'K: (none)')
    print(f'R: {response}')

## Step 3: Save adapter locally

The adapter is already saved to Google Drive at `data/decoder_adapter/` (~14 MB).

Download it to your local machine to use with `config.use_decoder = True`.

In [ ]:
# Check adapter size
import os
adapter_path = config.decoder_adapter_path
if adapter_path.exists():
    total = sum(f.stat().st_size for f in adapter_path.rglob('*') if f.is_file())
    print(f'Adapter saved at: {adapter_path}')
    print(f'Total size: {total / 1024 / 1024:.1f} MB')
    print(f'\nFiles:')
    for f in sorted(adapter_path.rglob('*')):
        if f.is_file():
            print(f'  {f.name}: {f.stat().st_size / 1024:.0f} KB')
else:
    print('Adapter not found. Training may have failed.')